In [ ]:
# the smallest possible baseline code for the meta-pool project


import random

from typing import Tuple
import matplotlib.pyplot as plt
import numpy as np


class CollaborativeLearningGroup:
    """A group of learners collaborating toward a common goal."""
    def __init__(self, learners):
        self.learners = learners

    def collaborate(self):
        """Allow learners to share experiences and reinforce learning."""
        for learner in self.learners:
            for peer in self.learners:
                if learner != peer:
                    learner.q_table += 0.1 * (peer.q_table - learner.q_table)


class DynaQAgent:
    """DynaQAgent with collaborative learning.

    Attributes:
    ----------
    n_states (int): Number of states in the environment.
    n_actions (int): Number of possible actions in the environment.
    epsilon (float): Probability of choosing a random action (exploration).
    alpha (float): Learning rate for updating Q-values.
    gamma (float): Discount factor for future rewards.
    planning_steps (int): Number of planning steps to perform.
    q_table (ndarray): Q-values table for state-action pairs.
    model (dict): Model to store state transitions and rewards.
    """

    def __init__(
        self,
        n_states: int,
        n_actions: int,
        epsilon: float = 0.1,
        alpha: float = 0.1,
        gamma: float = 0.95,
        planning_steps: int = 5,
    ) -> None:
        self.n_states = n_states
        self.n_actions = n_actions
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}

    def choose_action(self, state: int) -> int:
        if random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.n_actions)
        return np.argmax(self.q_table[state])

    def update(self, state: int, action: int, reward: float, next_state: int) -> None:
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state][best_next_action]
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += self.alpha * td_error
        self.model[(state, action)] = (reward, next_state)

        for _ in range(self.planning_steps):
            s, a = random.choice(list(self.model.keys()))
            r, s_next = self.model[(s, a)]
            best_next_a = np.argmax(self.q_table[s_next])
            td_target = r + self.gamma * self.q_table[s_next][best_next_a]
            td_error = td_target - self.q_table[s][a]
            self.q_table[s][a] += self.alpha * td_error


def simulate_collaborative_learning(agents, teacher, episodes=100):
    rewards = []
    for episode in range(episodes):
        for agent in agents:
            state = random.randint(0, agent.n_states - 1)
            action = agent.choose_action(state)
            reward = teacher.evaluate(state, action)
            next_state = (state + 1) % agent.n_states
            agent.update(state, action, reward, next_state)
        CollaborativeLearningGroup(agents).collaborate()
        rewards.append(np.mean([np.max(agent.q_table) for agent in agents]))
    return rewards

# Example usage
num_agents = 3
agents = [DynaQAgent(n_states=10, n_actions=4) for _ in range(num_agents)]
teacher = DynaQAgent(n_states=10, n_actions=4)
avg_rewards = simulate_collaborative_learning(agents, teacher)
print("Collaborative learning completed.")


In [ ]:
import random

from typing import Tuple
import matplotlib.pyplot as plt
import numpy as np

class CollaborativeLearningGroup:
    """A group of learners collaborating toward a common goal."""
    def __init__(self, learners):
        self.learners = learners

    def add_learner(self, new_learner):
        """Add a new learner to the group."""
        self.learners.append(new_learner)

    def collaborate(self):
        """Allow learners to share experiences and reinforce learning."""
        for learner in self.learners:
            for peer in self.learners:
                if learner != peer:
                    shared_experience = np.random.rand(*learner.q_table.shape) * learner.learning_speed
                    peer.q_table += 0.1 * shared_experience
                    learner.q_table += 0.05 * shared_experience


class DynaQAgent:
    """DynaQAgent with inferred skill differences 


    Attributes:
    ----------
    n_states (int): Number of states in the environment.
    n_actions (int): Number of possible actions in the environment.
    epsilon (float): Probability of choosing a random action (exploration).
    alpha (float): Learning rate for updating Q-values.
    gamma (float): Discount factor for future rewards.
    planning_steps (int): Number of planning steps to perform.
    q_table (ndarray): Q-values table for state-action pairs.
    model (dict): Model to store state transitions and rewards.
    """
    def __init__(
        self,
        n_states: int,
        n_actions: int,
        epsilon: float = 0.1,
        alpha: float = 0.1,
        gamma: float = 0.95,
        planning_steps: int = 5,
    ) -> None:
        self.n_states = n_states
        self.n_actions = n_actions
        self.epsilon = epsilon
        self.alpha = alpha
        self.gamma = gamma
        self.planning_steps = planning_steps
        self.q_table = np.zeros((n_states, n_actions))
        self.model = {}
        self.learning_speed = 1.0  # Default, inferred dynamically
        self.rewards_history = []

    def choose_action(self, state: int) -> int:
        if random.uniform(0, 1) < self.epsilon:
            return np.random.choice(self.n_actions)
        return np.argmax(self.q_table[state])

    def update(self, state: int, action: int, reward: float, next_state: int) -> None:
        self.rewards_history.append(reward)
        if len(self.rewards_history) > 50:
            self.rewards_history.pop(0)
        
        # Infer learning speed based on reward progression
        if len(self.rewards_history) > 10:
            improvement_rate = np.mean(self.rewards_history[-10:]) - np.mean(self.rewards_history[:10])
            self.learning_speed = max(0.5, min(1.5, 1.0 + improvement_rate))
        
        adjusted_alpha = self.alpha * self.learning_speed
        
        best_next_action = np.argmax(self.q_table[next_state])
        td_target = reward + self.gamma * self.q_table[next_state][best_next_action]
        td_error = td_target - self.q_table[state][action]
        self.q_table[state][action] += adjusted_alpha * td_error
        self.model[(state, action)] = (reward, next_state)

        for _ in range(self.planning_steps):
            s, a = random.choice(list(self.model.keys()))
            r, s_next = self.model[(s, a)]
            best_next_a = np.argmax(self.q_table[s_next])
            td_target = r + self.gamma * self.q_table[s_next][best_next_a]
            td_error = td_target - self.q_table[s][a]
            self.q_table[s][a] += adjusted_alpha * td_error


class SimulatedTeacher:
    """A class to simulate a teacher guiding a human learner."""
    def __init__(self, guidance_threshold=0.5):
        self.guidance_threshold = guidance_threshold

    def evaluate(self, state, action):
        return 1 if action in self.get_recommended_actions(state) else -1
    
    def get_recommended_actions(self, state):
        return [state % 2, (state + 1) % 2]  # More flexible recommendation

    def should_intervene(self, avg_reward: float) -> bool:
        return avg_reward < self.guidance_threshold

    def guide(self, learner):
        learner.q_table += 0.05 * np.random.rand(*learner.q_table.shape)


def simulate_collaborative_learning(agents, teacher, teacher_intervention=True, episodes=100):
    rewards = []
    group = CollaborativeLearningGroup(agents)
    
    for episode in range(episodes):
        for agent in agents:
            state = random.randint(0, agent.n_states - 1)
            action = agent.choose_action(state)
            reward = teacher.evaluate(state, action)
            next_state = (state + 1) % agent.n_states
            agent.update(state, action, reward, next_state)

            if teacher_intervention and teacher.should_intervene(np.mean(rewards[-10:]) if len(rewards) >= 10 else 1):
                teacher.guide(agent)
        
        group.collaborate()
        rewards.append(np.mean([np.max(agent.q_table) for agent in agents]))
    return rewards


def add_new_student(group, n_states=10, n_actions=4):
    """Interface to add a new student to the group dynamically."""
    new_student = DynaQAgent(n_states, n_actions)
    group.add_learner(new_student)
    print("New student added to the group.")


def evaluate_agents_performance(agents):
    """Evaluate and compare the performance of agents after the learning process."""
    for i, agent in enumerate(agents):
        best_action_values = np.max(agent.q_table, axis=1)
        avg_performance = np.mean(best_action_values)
        print(f"Agent {i+1} - Average performance: {avg_performance:.2f}")
        print(f"Best action values per state: {best_action_values[:5]}...")


num_agents = 3
agents = [DynaQAgent(n_states=10, n_actions=4) for _ in range(num_agents)]
teacher = SimulatedTeacher(guidance_threshold=0.5)
avg_rewards = simulate_collaborative_learning(agents, teacher, teacher_intervention=True)
print("Collaborative learning with teacher intervention completed.")

group = CollaborativeLearningGroup(agents)
add_new_student(group)

evaluate_agents_performance(agents)

# TODO: Instead of adding random values, experience sharing should be based on actual learning trajectories of the more skilled peer.
